<a href="https://colab.research.google.com/github/Prktriesitout/LangGraph/blob/main/Project5Langgraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Business Scenario:

An HR chatbot routes candidates based on their experience level.

Problem Statement:

Use conditional edges in LangGraph to:

Route to junior_node, mid_node, or senior_node based on experience_years.
 Steps to Follow:

Define state: name, experience_years, category.
Create decision node: classify_experience.
Add edges:
<3 → junior_node
3–7 → mid_node
>7 → senior_node
Test with multiple inputs.
 Expected Output:

Input: {name: 'Aditi', experience: 8}

Path: classify_experience → senior_node

Response: “You qualify for Senior Engineer role.”

The flow :-

START -> classify_experience -: 3 -> junior node -> END
                              3-7 -> mid node -> END
                              7 -> senior_node ->

Packages to Install

langgraph -> builds the graph, nodes  and conditional edges
langchain-core -> core types used by LangGraph
langchain-openai -> ChatOpenAI, the OpenAI connector

In [5]:
!pip install -q langgraph langchain-core langchain-openai


In [3]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OpenAI')

#

In [18]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI

class State(TypedDict):
  name:str
  experience_years:int
  category: str
  response: str

llm = ChatOpenAI(model="gpt-4o-mini",temperature=0)

def classify_experience(state: State) -> dict:
    years = state["experience_years"]
    if years < 3:
        category = "junior"
    elif years <= 7:
        category = "mid"
    else:
        category = "senior"
    print(f"classify_experience: {state['name']} has {years} years -> {category}")
    return {"category": category}

# sanity check: must print <class 'dict'>
r = classify_experience({"name": "Aditi", "experience_years": 8, "category": "", "response": ""})
print(type(r), r)



classify_experience: Aditi has 8 years -> senior
<class 'dict'> {'category': 'senior'}


In [10]:
def _respond(state: State, role:str):
  prompt = (
      f"You are an HR assistant. Write one short, friendly sentence telling "
      f"{state['name']} that they qualify for the {role} role. "
      f"They have {state['experience_years']} years of experience."
  )
  return {"response":  llm.invoke(prompt).content.strip()}


In [12]:
def junior_node(state: State):
  return _respond(state, "Junior Engineer")
def mid_node(state: State):
  return _respond(state, "Mid-Level Engineer")
def senior_node(state: State):
  return _respond(state, "Senior Engineer")


def route_by_category(state: State):
  return f"{state['category']}_node"

In [19]:
builder = StateGraph(State)
builder.add_node("classify_experience", classify_experience)
builder.add_node("junior_node", junior_node)
builder.add_node("mid_node", mid_node)
builder.add_node("senior_node", senior_node)

builder.add_edge(START, "classify_experience")
builder.add_conditional_edges(
    "classify_experience",
    route_by_category,
    {"junior_node": "junior_node", "mid_node": "mid_node", "senior_node": "senior_node"},
)
builder.add_edge("junior_node", END)
builder.add_edge("mid_node", END)
builder.add_edge("senior_node", END)

graph = builder.compile()

In [20]:
tests = [
    {"name": "Aditi", "experience_years": 8},
    {"name": "Rahul", "experience_years": 5},
    {"name": "Meera", "experience_years": 1},
    {"name": "Kiran", "experience_years": 3},
    {"name": "Arjun", "experience_years": 7},
]

for t in tests:
    print(f"\n===== Input: {t} =====")
    out = graph.invoke({**t, "category": "", "response": ""})
    print(f"Path: classify_experience -> {out['category']}_node")
    print(f"Response: {out['response']}")


===== Input: {'name': 'Aditi', 'experience_years': 8} =====
classify_experience: Aditi has 8 years -> senior
Path: classify_experience -> senior_node
Response: Hi Aditi! We're excited to let you know that with your 8 years of experience, you qualify for the Senior Engineer role!

===== Input: {'name': 'Rahul', 'experience_years': 5} =====
classify_experience: Rahul has 5 years -> mid
Path: classify_experience -> mid_node
Response: Hi Rahul! We're excited to let you know that with your 5 years of experience, you qualify for the Mid-Level Engineer role!

===== Input: {'name': 'Meera', 'experience_years': 1} =====
classify_experience: Meera has 1 years -> junior
Path: classify_experience -> junior_node
Response: Hi Meera! We're excited to let you know that you qualify for the Junior Engineer role with your one year of experience!

===== Input: {'name': 'Kiran', 'experience_years': 3} =====
classify_experience: Kiran has 3 years -> mid
Path: classify_experience -> mid_node
Response: Hi Ki